# 🤖 Build a Mini Transformer & Baby-GPT From Scratch (Day 1 Lab - ATTENDEE VERSION)
Welcome to the Day 1 Hands-On Workshop! Today, we will build the core engine of a Generative AI Transformer model from scratch using only **Python**, **NumPy**, and **PyTorch**.

### 🗺️ Our Coding Roadmap
```
Raw Text (list of sentences)
   ↓
Tokenizer (Convert text to lists of words & punctuation)
   ↓
Vocabulary (Build mapping of words to numerical IDs)
   ↓
Token IDs (Encode sentences into lists of numbers)
   ↓
Padding (Align all sequences to the same uniform length)
   ↓
Embeddings (Represent word meanings as high-dimensional vectors)
   ↓
Self-Attention (Play the Query-Key-Value matching game to capture relationships)
   ↓
Mini Transformer Block (Assemble Attention + Residuals + LayerNorm + Feedforward)
   ↓
Baby-GPT (Train a small PyTorch model on a corpus to auto-complete prompts!)
```

---

## 🧩 Part 1: The Tokenizer

Computers don't understand text natively. We need to split text into smaller units (tokens) and map them to numbers.

### 1.1 Simple Tokenization by Spacing
Let's look at a basic split tokenizer and see why it fails.

In [ ]:
raw_text = "I love AI, and AI loves me!"

def simple_tokenizer(text):
    text = text.lower()
    tokens = text.split()
    return tokens

tokens = simple_tokenizer(raw_text)
print("Tokens:", tokens)
# Note how 'ai,' and 'me!' include punctuation

### 1.2 Shared Vocabulary Mapping
Let's assign a unique number to each unique word.

In [ ]:
vocab = {}
for token in tokens:
    if token not in vocab:
        vocab[token] = len(vocab) + 1

print("Vocabulary Map:", vocab)

encoded_text = [vocab[token] for token in tokens]
print("Encoded Token IDs:", encoded_text)

### 1.3 Premium Regex Tokenizer & Sequence Padding
Real-world tokenizers must handle:
- Separating punctuation from words (`love,` -> `love` and `,`)
- Out-of-vocabulary words (`<UNK>`)
- Batches of sentences of different lengths (requiring `<PAD>` tokens)

#### ⚠️ blocker 1: Implement the vocabulary builder!
Complete the `_build_vocabulary` method below to assign a unique integer to each new word.

In [ ]:
import re
import numpy as np

class Tokenizer:
    def __init__(self, sentences: list[str]):
        self.sentences = sentences
        self.special_tokens = {"<PAD>": 0, "<UNK>": 1}
        self.tokenized_sentences = self._tokenize_all(sentences)
        self.vocab = self._build_vocabulary(self.tokenized_sentences)
        self.reverse_vocab = {idx: word for word, idx in self.vocab.items()}

    def _tokenize_sentence(self, text: str) -> list[str]:
        text = text.lower()
        # Use regex to separate alphanumeric words and individual punctuation marks
        tokens = re.findall(r'\w+|[^\w\s]', text)
        return tokens

    def _tokenize_all(self, sentences: list[str]) -> list[list[str]]:
        return [self._tokenize_sentence(s) for s in sentences]

    def _build_vocabulary(self, tokenized_sentences: list[list[str]]) -> dict[str, int]:
        vocab = dict(self.special_tokens) # start with pad & unk
        
        # ==================================================================
        # TODO: Blocker 1 - Loop through each sentence, and each token inside it.
        # If the token is NOT in 'vocab', add it to 'vocab' with value len(vocab).
        # ==================================================================
        # --- START YOUR CODE HERE ---
        pass
        # --- END YOUR CODE HERE ---
        
        return vocab

    def encoding(self, batch_sentences: list[str] = None) -> list[list[int]]:
        sentences_to_encode = self._tokenize_all(batch_sentences) if batch_sentences is not None else self.tokenized_sentences
        encoded = []
        for tokens in sentences_to_encode:
            ids = [self.vocab.get(token, self.vocab["<UNK>"]) for token in tokens]
            encoded.append(ids)
        return encoded

    def decoding(self, encoded_sentences: list[list[int]]) -> list[list[str]]:
        decoded = []
        for ids in encoded_sentences:
            words = [self.reverse_vocab.get(i, "<UNK>") for i in ids]
            decoded.append(words)
        return decoded

    def padding(self, encoded_sentences: list[list[int]] = None, max_length: int = None) -> list[list[int]]:
        if encoded_sentences is None:
            encoded_sentences = self.encoding()
        if max_length is None:
            max_length = max(len(s) for s in encoded_sentences)
        
        pad_id = self.vocab["<PAD>"]
        padded_sentences = []
        for sentence in encoded_sentences:
            padded = sentence[:max_length]
            padded = padded + [pad_id] * (max_length - len(padded))
            padded_sentences.append(padded)
        return padded_sentences

# Let's test our complete Tokenizer class
raw_sentences = [
    "I love AI, and AI loves me.",
    "https://google.com",
    "2026 workshop!",
    "if (x > 5):"
]

t = Tokenizer(raw_sentences)
print("Vocabulary Map:", t.vocab)
encoded = t.encoding()
print("\nEncoded IDs:", encoded)
padded = t.padding()
print("\nPadded ID Grid:\n", np.array(padded))

---

## 📚 Part 2: Word Embeddings

IDs like `3` or `12` are just indexes; they carry no meaning. A word embedding assigns a vector of numbers to each word, representing its semantic concept in a multi-dimensional space.

In [ ]:
class Embedding:
    def __init__(self, vocab_size, embedding_dim):
        # Initialize a random matrix of weights representing word vectors
        self.embedding_matrix = np.random.randn(vocab_size, embedding_dim)
    
    def embed(self, token_ids):
        # Extract vectors corresponding to token IDs
        return self.embedding_matrix[token_ids]

# Create an embedding layer: Vocab size from tokenizer, 4 dimensions per word
vocab_size = len(t.vocab)
embedding_dim = 4
emb_layer = Embedding(vocab_size, embedding_dim)

print("Embedding Matrix Shape (vocab_size, dimensions):", emb_layer.embedding_matrix.shape)
sample_padded = t.padding(max_length=8)
word_vectors = emb_layer.embed(sample_padded)
print("\nEmbeddings output shape (batch_size, sequence_length, dimensions):", word_vectors.shape)
print("\nWord vector for first sentence:\n", word_vectors[0])

---

## 🕵️‍♂️ Part 3: The Self-Attention Mechanism (The Magic Robot)

Self-attention allows words to look at other words in the sentence, calculate relationship strengths, and update their vectors to include context.

### 🤖 The Game of Q, K, and V
1. **Query (Q)**: "What friend am I looking for?"
2. **Key (K)**: "What friend am I?"
3. **Value (V)**: "What information do I contain?"

#### ⚠️ blocker 2: Calculate scaled similarity scores!
Complete the `forward` pass below to calculate matching scores and scale them by the square root of $d_k$.

In [ ]:
def softmax(x):
    # Standard Softmax calculation along the last axis
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

class SelfAttention:
    def __init__(self, d_model):
        # Initialize projection weight matrices for Query, Key, and Value
        self.Wq = np.random.randn(d_model, d_model)
        self.Wk = np.random.randn(d_model, d_model)
        self.Wv = np.random.randn(d_model, d_model)
        self.d_k = d_model
        
    def forward(self, x):
        # Step 1: Project input embeddings into Q, K, and V spaces
        Q = x @ self.Wq
        K = x @ self.Wk
        V = x @ self.Wv
        
        # ==================================================================
        # TODO: Blocker 2a - Calculate similarity scores by matrix-multiplying Q and K^T
        # Hint: K.swapaxes(-1, -2) yields the transpose on the last two axes. Use the @ operator.
        # ==================================================================
        scores = None 
        
        # ==================================================================
        # TODO: Blocker 2b - Scale down scores by dividing by the square root of self.d_k
        # Hint: Use np.sqrt(self.d_k)
        # ==================================================================
        scaled_scores = None
        
        # ==================================================================
        
        # Step 4: Turn scores into percentages (Softmax weights)
        attention_weights = softmax(scaled_scores)
        
        # Step 5: Weighted sum of Value vectors
        output = attention_weights @ V
        
        return output, attention_weights

# Test attention on our embeddings
# Note: It will raise an error until Blocker 2 is solved.
try:
    attention_layer = SelfAttention(d_model=embedding_dim)
    output_vectors, weights = attention_layer.forward(word_vectors)
    print("Input Vectors shape:", word_vectors.shape)
    print("Output Vectors shape:", output_vectors.shape)
    print("\nAttention Weights for sentence 1 (Relationship levels):\n", weights[0])
except Exception as e:
    print("Error:", e)

---

## 🏗️ Part 4: Building the Complete Mini Transformer Block

A full Transformer block wraps **Self-Attention** inside **Residual Connections**, **Layer Normalization**, and a **Feedforward Network**.

#### ⚠️ blocker 3: Add Residual Connections!
Implement the shortcut connections (`res1` and `res2`) inside the `forward` pass below to prevent vanishing gradients.

In [ ]:
def layer_norm(x, epsilon=1e-6):
    # Standard Layer Normalization: centers values around 0 with variance 1
    mean = np.mean(x, axis=-1, keepdims=True)
    variance = np.var(x, axis=-1, keepdims=True)
    return (x - mean) / np.sqrt(variance + epsilon)

class FeedForwardNetwork:
    def __init__(self, d_model, d_ff):
        # W1 expands dimensions; W2 projects them back
        self.W1 = np.random.randn(d_model, d_ff)
        self.W2 = np.random.randn(d_ff, d_model)
        
    def forward(self, x):
        # Expand dimensions, apply ReLU nonlinearity, and project back
        hidden = np.maximum(0, x @ self.W1) 
        return hidden @ self.W2

class MiniTransformerBlock:
    def __init__(self, d_model, d_ff):
        self.attention = SelfAttention(d_model)
        self.ffn = FeedForwardNetwork(d_model, d_ff)
        
    def forward(self, x):
        # --- BLOCK 1: Attention, Add & Norm ---
        attn_out, weights = self.attention.forward(x)
        
        # ==================================================================
        # TODO: Blocker 3a - Add the residual connection (Add original input 'x' to 'attn_out')
        # ==================================================================
        res1 = None
        norm1 = layer_norm(res1)
        
        # --- BLOCK 2: FeedForward, Add & Norm ---
        ffn_out = self.ffn.forward(norm1)
        
        # ==================================================================
        # TODO: Blocker 3b - Add the second residual connection (Add 'norm1' to 'ffn_out')
        # ==================================================================
        res2 = None
        final_out = layer_norm(res2)
        
        return final_out, weights

# Test the complete block!
try:
    transformer_block = MiniTransformerBlock(d_model=embedding_dim, d_ff=8)
    final_embeddings, final_weights = transformer_block.forward(word_vectors)
    print("Success! Our block outputs values.")
    print("Final output shape:", final_embeddings.shape)
except Exception as e:
    print("Error:", e)

---

## 🚀 Part 5: End-to-End Project: Build & Train "Baby-GPT"

Now we are going to combine all these blocks in **PyTorch** to build a generative next-word text predictor. We will train it on a mini-corpus, and then chat with it by giving it prompts!

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Define our training text corpus
corpus = [
    "i love ai and ai loves me",
    "deep learning is powerful and ai is the future",
    "transformers are the core of generative ai models",
    "we are coding a mini gpt model today",
    "ai models generate text by predicting the next token"
]

# Initialize our custom tokenizer on the corpus
# Note: This depends on Blocker 1 being complete!
gpt_tokenizer = Tokenizer(corpus)
vocab_size = len(gpt_tokenizer.vocab)
print("GPT Vocabulary size:", vocab_size)
print("Vocabulary map:\n", gpt_tokenizer.vocab)

### 5.2 Build the Baby-GPT Model in PyTorch
We will translate our NumPy modules into PyTorch layers so we can utilize automatic differentiation (autograd) to train the model weights!

In [ ]:
class PyTorchSelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.d_k = d_model
        
    def forward(self, x):
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)
        
        # Similarity matrix: Q @ K^T
        scores = torch.matmul(Q, K.transpose(-1, -2))
        scaled_scores = scores / (self.d_k ** 0.5)
        weights = F.softmax(scaled_scores, dim=-1)
        return torch.matmul(weights, V)

class PyTorchTransformerBlock(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.attention = PyTorchSelfAttention(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
    def forward(self, x):
        # Self-Attention + Residual Add + LayerNorm
        attn_out = self.attention(x)
        x = self.norm1(x + attn_out)
        
        # Feedforward + Residual Add + LayerNorm
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x

class BabyGPT(nn.Module):
    def __init__(self, vocab_size, d_model, d_ff, max_seq_len):
        super().__init__()
        self.token_embeddings = nn.Embedding(vocab_size, d_model)
        self.position_embeddings = nn.Embedding(max_seq_len, d_model)
        self.transformer = PyTorchTransformerBlock(d_model, d_ff)
        self.lm_head = nn.Linear(d_model, vocab_size) # Maps representations back to vocabulary tokens
        self.max_seq_len = max_seq_len
        
    def forward(self, idx):
        batch_size, seq_len = idx.shape
        # Position ids: [0, 1, 2, ... seq_len - 1]
        pos = torch.arange(0, seq_len, dtype=torch.long, device=idx.device)
        
        # Sum word meanings + positional information
        tok_emb = self.token_embeddings(idx)
        pos_emb = self.position_embeddings(pos)
        x = tok_emb + pos_emb
        
        x = self.transformer(x)
        logits = self.lm_head(x)
        return logits

### 5.3 Prepare Training Sequences
For next-token prediction, if our sequence is `[i, love, ai]`, the inputs and targets shift by one:
- **Inputs**: `[i, love]`
- **Targets**: `[love, ai]`

In [ ]:
# Encode and pad our corpus to generate tensor datasets
encoded_corpus = gpt_tokenizer.encoding()
max_seq_len = max(len(s) for s in encoded_corpus)
padded_corpus = gpt_tokenizer.padding(encoded_corpus, max_length=max_seq_len)

padded_array = np.array(padded_corpus)

# Inputs: all tokens except the very last one
X_train = torch.tensor(padded_array[:, :-1], dtype=torch.long)
# Targets: shifted by one token to the right
Y_train = torch.tensor(padded_array[:, 1:], dtype=torch.long)

print("Training Input shape:", X_train.shape)
print("Training Target shape:", Y_train.shape)
print("\nFirst Sentence Example:")
print("  Input IDs: ", X_train[0].tolist())
print("  Target IDs:", Y_train[0].tolist())

### 5.4 Execute the Training Loop
We will run gradient descent optimization using the Adam optimizer and Cross Entropy loss.

In [ ]:
import torch.optim as optim

d_model = 16
d_ff = 32
epochs = 150

model = BabyGPT(vocab_size, d_model, d_ff, max_seq_len)
criterion = nn.CrossEntropyLoss(ignore_index=0) # Ignore <PAD> tokens in loss calculation
optimizer = optim.Adam(model.parameters(), lr=0.01)

print("Training the Baby-GPT Model...")
model.train()
for epoch in range(epochs):
    optimizer.zero_grad()
    logits = model(X_train)
    
    # Flatten outputs and targets for Cross Entropy calculations
    loss = criterion(logits.view(-1, vocab_size), Y_train.view(-1))
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 25 == 0:
        print(f"  Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")

### 5.5 Autoregressive Text Generation (Prompting)
Now we can give the model a custom prompt and see how it generates text.

In [ ]:
def generate_text(model, prompt: str, max_new_tokens: int = 5) -> str:
    model.eval()
    words = prompt.lower().split()
    
    print(f"Prompt: '{prompt}'")
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            # Encode and pad current words
            encoded = gpt_tokenizer.encoding([" ".join(words)])
            padded = gpt_tokenizer.padding(encoded, max_length=max_seq_len - 1)
            idx_tensor = torch.tensor(padded, dtype=torch.long)
            
            # Forward pass: get logit predictions
            logits = model(idx_tensor)
            
            # Extract logits for the last token position
            seq_length_of_words = len(gpt_tokenizer._tokenize_sentence(" ".join(words)))
            last_token_logits = logits[0, min(seq_length_of_words - 1, max_seq_len - 2), :]
            
            # Predict token ID with the highest probability (greedy sampling)
            predicted_id = torch.argmax(last_token_logits).item()
            
            # Decode predicted ID to word string
            predicted_word = gpt_tokenizer.reverse_vocab.get(predicted_id, "<UNK>")
            
            if predicted_word == "<PAD>":
                break
            words.append(predicted_word)
            
    return " ".join(words)

# Test generate with our trained Baby-GPT!
try:
    prompt1 = "i love"
    output1 = generate_text(model, prompt1, max_new_tokens=4)
    print("Output:", output1)
    
    print("-" * 50)
    prompt2 = "transformers are"
    output2 = generate_text(model, prompt2, max_new_tokens=5)
    print("Output:", output2)
except Exception as e:
    print("Error:", e)